# Amazon's Scrapped Hiring Tool: When a Word Becomes a Proxy

In this notebook I look at Amazon's experimental resume-screening tool, reported to have been scrapped around 2017 to 2018, as a case study in a different mechanism than the previous two notebooks in this series.

The Tokyo Medical University notebook was about a deliberate, manual adjustment. The COMPAS notebook was about an honestly calibrated score that still produced unequal error rates because two groups had different base rates. This one is about something that needs neither a deliberate adjustment nor a calibration paradox: a feature that looks entirely neutral, on its own, can still stand in for a protected attribute, if it happens to correlate with group membership in the historical data a system is trained on.

In this notebook, I will:

- Summarize what was reported about the tool
- Build a synthetic resume feature that correlates with group membership but has nothing to do with qualification
- Simulate biased historical hiring decisions, the kind of labels a real system would have been trained on
- Fit a simple scoring rule to those labels, without ever giving it group membership as a feature, and see what it learns anyway
- Apply that rule to brand-new applicants and see whether removing the one proxy feature actually fixes it

## 1. Background: What Was Reported

Amazon built an experimental recruiting engine intended to review resumes and score candidates automatically, training the system on roughly a decade of resumes submitted to the company, a period during which the tech industry, and Amazon's own applicant pool, was heavily male.

According to reporting on the project, the system taught itself that resumes containing certain patterns associated with women were less desirable, including penalizing resumes that used the word "women's", as in a club or society name, and downgrading graduates of two all-women's colleges. Engineers reportedly edited the system to stop it from weighting those specific patterns, but could not be confident it would not find other, similarly biased proxies on its own. The project was reportedly scrapped before it was used to evaluate real candidates.

## 2. Why a Neutral-Looking Feature Can Still Encode Bias

Nobody had to tell this kind of system anyone's gender for it to end up sorting on it. If a word or phrase happens to show up far more often on resumes from one group than another, for reasons that have nothing to do with qualification, a system trained to match historical outcomes can learn to key off that word as a stand-in for the group itself. Removing gender as an explicit input field does nothing to stop this, because the system was never using a field called gender in the first place, it was using whatever correlated with the outcomes it was shown.

## 3. A Note on the Simulation Below

Everything from here on is a synthetic simulation using two unlabeled groups, Group A and Group B, not the real dataset or the real feature Amazon's system reportedly keyed on. The goal is to make the underlying mechanism, a neutral feature acting as a proxy, concrete enough to reason about, not to reproduce the specifics of the real system.

## 4. Simulating Applicants With a Group-Correlated Feature

Each applicant gets a qualification score drawn from the same distribution for both groups, so nobody starts out more or less qualified on average. Each applicant also gets a `keyword_flag`, standing in for some phrase or pattern in a resume that happens to show up far more often for one group than the other, entirely independent of qualification.

In [ ]:
import random

random.seed(11)


def simulate_applicants(count):
    """Returns a list of applicant dicts with a group, qualification, and keyword flag."""
    applicants = []
    for _ in range(count):
        group = "A" if random.random() < 0.5 else "B"
        qualification = max(0, min(100, random.gauss(70, 15)))
        keyword_probability = 0.1 if group == "A" else 0.9
        keyword_flag = random.random() < keyword_probability
        applicants.append({
            "group": group,
            "qualification": qualification,
            "keyword_flag": keyword_flag,
        })
    return applicants

## 5. Checking the Keyword Flag's Correlation With Group

Before going further, I confirm the flag actually behaves the way I intended: common in one group, rare in the other, and unrelated to qualification.

In [ ]:
sample = simulate_applicants(2000)


def keyword_rate(population, group):
    """Returns the fraction of a group that has the keyword flag set."""
    filtered = [a for a in population if a["group"] == group]
    flagged = [a for a in filtered if a["keyword_flag"]]
    return len(flagged) / len(filtered)


print("Keyword rate, group A:", round(keyword_rate(sample, "A"), 3))
print("Keyword rate, group B:", round(keyword_rate(sample, "B"), 3))

## 6. Simulating Biased Historical Hiring Decisions

Now I generate the training labels a real system like this would have learned from: past human hiring decisions. I model those decisions as applying a higher qualification bar to group B than to group A, standing in for the kind of historical human bias that a resume-screening tool trained on past outcomes would simply absorb as ground truth.

In [ ]:
def historical_hire_label(applicant):
    """Returns the historical hire decision, using a group-dependent bar."""
    bar = 75 if applicant["group"] == "A" else 85
    return applicant["qualification"] >= bar


historical_applicants = simulate_applicants(3000)
for applicant in historical_applicants:
    applicant["hired"] = historical_hire_label(applicant)

## 7. Checking the Historical Hire Rate Gap

I check that this actually produced a hiring gap between the two groups, since this gap is what the model in the next step will be trained to reproduce.

In [ ]:
def hire_rate(population, group):
    """Returns the fraction of a group that was historically hired."""
    filtered = [a for a in population if a["group"] == group]
    hired = [a for a in filtered if a["hired"]]
    return len(hired) / len(filtered)


print("Historical hire rate, group A:", round(hire_rate(historical_applicants, "A"), 3))
print("Historical hire rate, group B:", round(hire_rate(historical_applicants, "B"), 3))

## 8. Fitting a Simple Scoring Rule to the Historical Decisions

Now I fit a small stand-in for a trained model: a score built from qualification plus a learned penalty applied when the keyword flag is set, compared against a learned threshold. Group is never given to this fitting process as an input, only qualification, the keyword flag, and the historical hire label it is trying to match. I search over candidate penalties and thresholds for the combination that best reproduces the historical decisions.

In [ ]:
def predict_hire(applicant, keyword_penalty, threshold):
    """Predicts a hire decision from qualification and the keyword flag only."""
    score = applicant["qualification"]
    if applicant["keyword_flag"]:
        score += keyword_penalty
    return score >= threshold


def accuracy(population, keyword_penalty, threshold):
    """Returns how often the rule matches the historical hire label."""
    correct = sum(
        1 for a in population
        if predict_hire(a, keyword_penalty, threshold) == a["hired"]
    )
    return correct / len(population)


best_penalty, best_threshold, best_accuracy = None, None, -1
for penalty in range(-40, 1):
    for threshold in range(50, 100):
        acc = accuracy(historical_applicants, penalty, threshold)
        if acc > best_accuracy:
            best_penalty, best_threshold, best_accuracy = penalty, threshold, acc

## 9. Checking What the Fitted Rule Learned

I print the penalty and threshold the search landed on, along with how well the rule matches the historical decisions.

In [ ]:
print("Learned keyword penalty:", best_penalty)
print("Learned threshold:", best_threshold)
print("Match with historical decisions:", round(best_accuracy, 3))

## 10. Applying the Fitted Rule to Fresh Applicants

Now I generate a brand-new pool of applicants, drawn from the exact same fair qualification distribution as before, with the same natural keyword correlation, but with no historical hire label attached to any of them, these are new candidates the rule has never seen. I check the rule's predicted hire rate for each group.

In [ ]:
new_applicants = simulate_applicants(3000)


def predicted_hire_rate(population, group, keyword_penalty, threshold):
    """Returns the rule's predicted hire rate for one group."""
    filtered = [a for a in population if a["group"] == group]
    hired = [a for a in filtered if predict_hire(a, keyword_penalty, threshold)]
    return len(hired) / len(filtered)


print(
    "Predicted hire rate, group A:",
    round(predicted_hire_rate(new_applicants, "A", best_penalty, best_threshold), 3),
)
print(
    "Predicted hire rate, group B:",
    round(predicted_hire_rate(new_applicants, "B", best_penalty, best_threshold), 3),
)